# धडा ०८ - मल्टि-एजंट डिझाईन पॅटर्न


## सेटअप


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## मल्टि-एजंट सिस्टीम का?

वास्तवातील कामं जसे ट्रिप नियोजन यामध्ये अनेक प्रकारच्या कौशल्याची गरज असते — लॉजिस्टिक्स, स्थानिक माहिती, बजेटिंग, आणि बरेच काही. सर्व काही हाताळण्याचा प्रयत्न करणारा एक एकल एजंट पटकन कंट्रोलशिवाय होतो.

मल्टि-एजंट सिस्टीम हे **विशेषीकरण** द्वारे हे सोडवतात: प्रत्येक एजंट एका विशिष्ट क्षेत्रावर लक्ष केंद्रित करतो, ज्यामुळे सामान्य एजंटपेक्षा उच्च दर्जाच्या निकालांची निर्मिती होते. ते **स्केलेबिलिटी** सुध्दा सुधारित करतात — तुम्ही नवीन एजंट जोडू शकता (उदा., फ्लाइट स्पेशालिस्ट, रेस्टॉरंट क्रिटिक) विद्यमान कार्यप्रवाह पुन्हा लिहिण्याशिवाय. एजंट एकत्रितपणे संरचित पाईपलाइनद्वारे कार्य करतात, एकमेकांकडून संदर्भ पुढे पास करतं.


## विशेष एजंट तयार करणे


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## क्रमवार वर्कफ्लो तयार करणे

`WorkflowBuilder` तुम्हाला एजंट्सना निर्देशित ग्राफमध्ये जोडण्याची परवानगी देतो. येथे आपण एक सोपा दोन-पायरी पाइपलाइन तयार करतो: **TravelPlanner** प्रवासाचा आराखडा तयार करतो, नंतर **TravelConcierge** त्याचा आढावा घेते आणि सुधारणा करते.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## वर्कफ्लोमध्ये अधिक एजंट्स जोडणे

मल्टी-एजंट पॅटर्नचे सर्वात मोठे फायदे म्हणजे त्याचा विस्तार करणे किती सोपे आहे. खाली आपण एक **BudgetReviewer** एजंट जोडतो जो प्रवाशाच्या बजेटशी योजना तपासतो, खर्च मर्यादेपेक्षा जास्त होऊ शकणाऱ्या गोष्टींवर ध्येय देतो आणि पैसे वाचविण्यासाठी पर्याय सुचवतो. वर्कफ्लो आता तीन एजंट्स अनुक्रमे चालवतो:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## सारांश

या धड्यात आपण शिकले कसे:

1. **विशेष एजंट तयार करणे** — प्रत्येकाचा एक लक्षित भूमिका (योजना, कंसीअर्ज, बजेट परीक्षण).
2. **एजंट्सना अनुक्रमिक कार्यप्रवाहात जोडणे** `WorkflowBuilder` आणि `add_edge` वापरून.
3. **मल्टी-एजंट पाइपलाइनमधून आउटपुट प्रवाहित करणे**, कोणता एजंट बोलत आहे हे ट्रॅक करत.
4. **कार्यप्रवाह विस्तृत करणे** नवीन एजंट्स साखळीमध्ये जोडून विद्यमान एजंट न बदलता.

मल्टी-एजंट डिझाइन नमुना प्रत्येक एजंटला सोपा ठेवतो तर एकटे एजंट साधू शकतात त्यापेक्षा जास्त समृद्ध, अधिक तपासलेले परिणाम तयार करतो.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
